In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/kyc.parquet
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/train_labels.csv
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/test.csv
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/dayend_balance/balance_2024-01.parquet
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/dayend_balance/balance_2024-03.parquet
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/dayend_balance/balance_2024-02.parquet
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/transactions/trx_2024-03.parquet
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/transactions/trx_2024-01.parquet
/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public/transactions/t

In [ ]:

import subprocess
subprocess.run(["pip", "install", "lifelines", "--quiet"], check=True)
print("lifelines installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.0 MB/s eta 0:00:00
lifelines installed


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from lifelines.utils import concordance_index

BASE = "/kaggle/input/datasets/mohibulhasantarek/bkash-presents-nsucec-datathon-final/public"
TRX  = f"{BASE}/transactions/trx_2024-0[1-3].parquet"
BAL  = f"{BASE}/dayend_balance/balance_2024-0[1-3].parquet"
REF  = pl.datetime(2024, 3, 31)   # observation cutoff

labels = pl.read_csv(f"{BASE}/train_labels.csv")
test   = pl.read_csv(f"{BASE}/test.csv")
kyc    = pl.read_parquet(f"{BASE}/kyc.parquet")

print("labels:", labels.shape, labels.columns)
print("test:  ", test.shape)
print("kyc:   ", kyc.shape, kyc.columns)

labels: (595000, 3) ['ACCOUNT_ID', 'DURATION_DAYS', 'EVENT_FLAG']
test:   (255000, 1)
kyc:    (1000000, 5) ['ACCOUNT_ID', 'ACCOUNT_TYPE', 'ACCOUNT_OPEN_DATE', 'GENDER', 'REGION']


In [ ]:
#feature engineering
trx_base = (
    pl.scan_parquet(TRX)
      .with_columns(pl.col("TRX_DATETIME").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("dt"))
      .group_by("SRC_ACCOUNT")
      .agg([
          (REF - pl.col("dt").max()).dt.total_days().alias("recency_days"),
          pl.len().alias("freq_total"),
          (pl.col("dt") >= pl.datetime(2024, 3, 1)).sum().alias("freq_30d"),
          (pl.col("dt") >= pl.datetime(2024, 3, 24)).sum().alias("freq_7d"),
          ((pl.col("dt") >= pl.datetime(2024, 1, 31)) &
           (pl.col("dt") <  pl.datetime(2024, 3,  1))).sum().alias("freq_prev30"),
          pl.col("TRX_AMT").sum().alias("total_amt"),
          (pl.col("dt").max() - pl.col("dt").min()).dt.total_days().alias("active_span"),
          # gap features
          pl.col("dt").sort().diff().dt.total_days().max().alias("max_gap_days"),
          pl.col("dt").sort().diff().dt.total_days().mean().alias("mean_gap_days"),
          (pl.col("dt").sort().diff().dt.total_days() >= 20).sum().alias("n_gaps_ge20"),
          (pl.col("dt").sort().diff().dt.total_days() >= 25).sum().alias("n_gaps_ge25"),
      ])
      .with_columns(
          (pl.col("freq_30d") / (pl.col("freq_total") + 1)).alias("activity_concentration")
      )
      .rename({"SRC_ACCOUNT": "ACCOUNT_ID"})
      .collect(engine="streaming")
)
print("trx_base:", trx_base.shape)

bal_feats = (
    pl.scan_parquet(BAL)
      .with_columns(pl.col("DATE").str.to_date("%Y-%m-%d").alias("d"))
      .sort(["ACCOUNT_ID", "d"])
      .with_columns(
          (pl.col("AVAILABLE_BALANCE").diff().over("ACCOUNT_ID").abs() > 0).alias("changed")
      )
      .group_by("ACCOUNT_ID")
      .agg([
          pl.col("AVAILABLE_BALANCE").sort_by("d").last().alias("bal_last"),
          pl.col("AVAILABLE_BALANCE").max().alias("bal_max"),
          pl.col("AVAILABLE_BALANCE").filter(
              pl.col("d") >= pl.date(2024, 3, 17)).std().alias("bal_std_14d"),
          (pl.col("changed") & (pl.col("d") >= pl.date(2024, 3, 17))).sum().alias("bal_changes_14d"),
          (pl.date(2024, 3, 31) - pl.col("d").filter(
              pl.col("changed")).max()).dt.total_days().alias("days_since_bal_change"),
      ])
      .with_columns(
          (1 - (pl.col("bal_last") / (pl.col("bal_max") + 1))).alias("bal_drawdown")
      )
      .collect(engine="streaming")
)
print("bal_feats:", bal_feats.shape)


tenure = (
    kyc
      .with_columns((REF - pl.col("ACCOUNT_OPEN_DATE")).dt.total_days().alias("tenure_days"))
      .select(["ACCOUNT_ID", "tenure_days"])
)
print("tenure:", tenure.shape)

reg_feats = (
    pl.scan_parquet(TRX)
      .with_columns(pl.col("TRX_DATETIME").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("dt"))
      .with_columns(pl.col("dt").dt.date().alias("day"))
      .group_by("SRC_ACCOUNT")
      .agg([
          pl.len().alias("n_txn"),
          pl.col("day").n_unique().alias("n_active_days"),
          pl.col("dt").sort().diff().dt.total_hours().std().alias("gap_std_h"),
          pl.col("dt").sort().diff().dt.total_hours().mean().alias("gap_mean_h"),
          pl.col("TRX_AMT").std().alias("amt_std"),
          pl.col("TRX_AMT").mean().alias("amt_mean"),
          (pl.col("dt") < pl.datetime(2024, 2, 1)).sum().alias("m_jan"),
          (pl.col("dt") >= pl.datetime(2024, 3, 1)).sum().alias("m_mar"),
      ])
      .with_columns([
          (pl.col("gap_std_h") / (pl.col("gap_mean_h") + 1)).alias("gap_cv"),
          (pl.col("amt_std")   / (pl.col("amt_mean")   + 1)).alias("amt_cv"),
          (pl.col("n_active_days") / (pl.col("n_txn") + 1)).alias("spread_ratio"),
          (pl.col("m_mar") / (pl.col("m_jan") + 1)).alias("mar_jan_ratio"),
      ])
      .rename({"SRC_ACCOUNT": "ACCOUNT_ID"})
      .collect(engine="streaming")
)
print("reg_feats:", reg_feats.shape)

edges = (
    pl.scan_parquet(TRX)
      .group_by(["SRC_ACCOUNT", "DST_ACCOUNT"]).agg(pl.len().alias("w"))
      .collect(engine="streaming")
)
net_feats = (
    edges.lazy()
      .group_by("SRC_ACCOUNT")
      .agg([
          pl.col("DST_ACCOUNT").n_unique().alias("n_unique_dst"),
          pl.col("w").sum().alias("tot_w"),
          pl.col("w").max().alias("max_w"),
          (-(( pl.col("w") / pl.col("w").sum()) *
             ( pl.col("w") / pl.col("w").sum()).log()).sum()).alias("cp_entropy"),
      ])
      .with_columns([
          (pl.col("tot_w") / (pl.col("n_unique_dst") + 1)).alias("txn_per_cp"),
          (pl.col("max_w") / (pl.col("tot_w")       + 1)).alias("top_cp_share"),
      ])
      .rename({"SRC_ACCOUNT": "ACCOUNT_ID"})
      .collect()
)
typemix = (
    pl.scan_parquet(TRX)
      .group_by("SRC_ACCOUNT")
      .agg([
          (pl.col("TRX_TYPE") == "P2P").mean().alias("p2p_share"),
          (pl.col("TRX_TYPE").is_in(["MerchantPay", "BillPay"])).mean().alias("merchant_share"),
          (pl.col("TRX_TYPE") == "CashOut").mean().alias("cashout_share"),
          pl.col("TRX_TYPE").n_unique().alias("n_trx_types"),
      ])
      .rename({"SRC_ACCOUNT": "ACCOUNT_ID"})
      .collect(engine="streaming")
)
net_all = net_feats.join(typemix, on="ACCOUNT_ID", how="left")
print("net_all:", net_all.shape)

amt_feats = (
    pl.scan_parquet(TRX)
      .with_columns(pl.col("TRX_DATETIME").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("dt"))
      .group_by("SRC_ACCOUNT")
      .agg([
          (pl.col("TRX_AMT").filter(pl.col("dt") >= pl.datetime(2024, 3, 1)).mean()
           / (pl.col("TRX_AMT").filter(pl.col("dt") < pl.datetime(2024, 3, 1)).mean() + 1)
          ).alias("amt_trend_mar"),
          (pl.col("TRX_AMT") > pl.col("TRX_AMT").mean() * 3).mean().alias("big_txn_share"),
          (pl.col("TRX_AMT") % 100 == 0).mean().alias("round_amt_share"),
          pl.col("TRX_AMT").log1p().std().alias("log_amt_std"),
          pl.col("TRX_AMT").max().alias("amt_max"),
      ])
      .rename({"SRC_ACCOUNT": "ACCOUNT_ID"})
      .collect(engine="streaming")
)
print("amt_feats:", amt_feats.shape)

trx_base: (849690, 13)
bal_feats: (850000, 7)
tenure: (1000000, 2)
reg_feats: (849690, 13)
net_all: (849690, 11)
amt_feats: (849690, 6)


In [ ]:

ZERO_COLS = [
    "freq_total", "freq_30d", "freq_7d", "freq_prev30", "total_amt",
    "activity_concentration", "active_span", "bal_changes_14d",
    "max_gap_days", "mean_gap_days", "n_gaps_ge20", "n_gaps_ge25",
]

def assemble(base_pl):
    """Join all feature tables onto a base Polars frame (labels or test)."""
    REG  = ["n_active_days", "gap_cv", "amt_cv", "spread_ratio", "mar_jan_ratio"]
    NET  = ["n_unique_dst", "cp_entropy", "txn_per_cp", "top_cp_share",
            "p2p_share", "merchant_share", "cashout_share", "n_trx_types"]
    AMT  = ["amt_trend_mar", "big_txn_share", "round_amt_share", "log_amt_std", "amt_max"]

    df = (
        base_pl
          .join(trx_base,  on="ACCOUNT_ID", how="left")
          .join(bal_feats, on="ACCOUNT_ID", how="left")
          .join(tenure,    on="ACCOUNT_ID", how="left")
          .join(reg_feats.select(["ACCOUNT_ID"] + REG), on="ACCOUNT_ID", how="left")
          .join(net_all.select(["ACCOUNT_ID"] + NET),   on="ACCOUNT_ID", how="left")
          .join(amt_feats,  on="ACCOUNT_ID", how="left")
          # fill nulls
          .with_columns([pl.col(c).fill_null(0) for c in ZERO_COLS])
          .with_columns([pl.col(c).fill_null(0) for c in REG + NET + AMT])
          .with_columns([
              pl.col("recency_days").fill_null(91),
              pl.col("bal_last").fill_null(0),
              pl.col("bal_max").fill_null(0),
              pl.col("bal_std_14d").fill_null(0),
              pl.col("bal_drawdown").fill_null(1.0),
              pl.col("days_since_bal_change").fill_null(91),
              pl.col("tenure_days").fill_null(pl.col("tenure_days").median()),
          ])
          .to_pandas()
    )
    
    df["true_max_gap"]   = df[["max_gap_days", "recency_days"]].max(axis=1)
    df["gap_to_30_ratio"] = df["true_max_gap"] / 30.0
    df["decel_week"]     = df["freq_7d"]  / (df["freq_30d"]   + 1)
    df["decel_month"]    = df["freq_30d"] / (df["freq_prev30"] + 1)
    df["gap_vs_mean"]    = df["true_max_gap"] / (df["mean_gap_days"] + 1)
    return df

train_df = assemble(labels)
test_df  = assemble(test)


BASE_FEATS = [
    "recency_days", "freq_total", "freq_30d", "freq_7d", "freq_prev30",
    "total_amt", "activity_concentration", "active_span",
    "bal_last", "bal_max", "bal_std_14d", "bal_changes_14d",
    "days_since_bal_change", "bal_drawdown", "tenure_days",
    # gap
    "mean_gap_days", "n_gaps_ge20", "n_gaps_ge25", "true_max_gap", "gap_to_30_ratio",
    # decel
    "decel_week", "decel_month", "gap_vs_mean",
]
REG_FEATS = ["n_active_days", "gap_cv", "amt_cv", "spread_ratio", "mar_jan_ratio"]
NET_FEATS = ["n_unique_dst", "cp_entropy", "txn_per_cp", "top_cp_share",
             "p2p_share", "merchant_share", "cashout_share", "n_trx_types"]
AMT_FEATS = ["amt_trend_mar", "big_txn_share", "round_amt_share", "log_amt_std", "amt_max"]

ALL_FEATS = BASE_FEATS + REG_FEATS + NET_FEATS + AMT_FEATS

# sanity checks
assert train_df.shape[0] == 595000 and test_df.shape[0] == 255000
assert train_df[ALL_FEATS].isnull().sum().sum() == 0
assert test_df[ALL_FEATS].isnull().sum().sum() == 0

dur = train_df["DURATION_DAYS"].values
evt = train_df["EVENT_FLAG"].values

print(f"train_df: {train_df.shape}  |  test_df: {test_df.shape}")
print(f"Total features: {len(ALL_FEATS)}")
print(f"  BASE={len(BASE_FEATS)}  REG={len(REG_FEATS)}  NET={len(NET_FEATS)}  AMT={len(AMT_FEATS)}")
print(f"Event rate: {evt.mean():.4f}")

train_df: (595000, 45)  |  test_df: (255000, 43)
Total features: 41
  BASE=23  REG=5  NET=8  AMT=5
Event rate: 0.0521


In [ ]:
Xtr = train_df[ALL_FEATS].values
Xte = test_df[ALL_FEATS].values

PARAMS = dict(
    objective="binary",
    n_estimators=1800,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.7,
    subsample_freq=1,         
    colsample_bytree=0.7,
    reg_lambda=5.0,
    min_child_samples=100,
    random_state=7,
    n_jobs=-1,
    verbose=-1,
)

#dola 

skf  = StratifiedKFold(5, shuffle=True, random_state=7)
oof  = {h: np.zeros(len(Xtr)) for h in [30, 60, 90]}
pred = {h: np.zeros(len(Xte)) for h in [30, 60, 90]}

for h in [30, 60, 90]:
    yb = ((evt == 1) & (dur <= h)).astype(int)
    print(f"  horizon {h:2d}d  |  positives: {yb.sum():,}")
    for tr, va in skf.split(Xtr, yb):
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(
            Xtr[tr], yb[tr],
            eval_set=[(Xtr[va], yb[va])],
            callbacks=[lgb.early_stopping(80, verbose=False)],
        )
        oof[h][va]  = m.predict_proba(Xtr[va])[:, 1]
        pred[h]    += m.predict_proba(Xte)[:, 1] / 5


Foof  = np.maximum.accumulate(np.vstack([oof[30],  oof[60],  oof[90]]).T,  axis=1)
Ftest = np.maximum.accumulate(np.vstack([pred[30], pred[60], pred[90]]).T, axis=1)

ci    = concordance_index(dur, -Foof.sum(1), evt)
brier = [np.mean((Foof[:, i] - ((evt == 1) & (dur <= h)))**2)
         for i, h in enumerate([30, 60, 90])]

print(f"\nOOF c-index : {ci:.4f}")
print(f"OOF Brier   : {[round(b,5) for b in brier]}  |  mean: {np.mean(brier):.5f}")
print(f"Est pts     : {round(max(0,(ci-.5)/.5)*40,1)} c-index  +  "
      f"{round(max(0,(.05-np.mean(brier))/.05)*30,1)} IBS")

  horizon 30d  |  positives: 286
  horizon 60d  |  positives: 1,879
  horizon 90d  |  positives: 30,994

OOF c-index : 0.7638
OOF Brier   : [np.float64(0.00048), np.float64(0.00308), np.float64(0.0464)]  |  mean: 0.01665
Est pts     : 21.1 c-index  +  20.0 IBS


In [ ]:
surv = 1 - Ftest   

sub = pd.DataFrame({
    "ACCOUNT_ID"   : test_df["ACCOUNT_ID"].values,
    "RISK_SCORE"   : Ftest.sum(1),
    "SURV_PROB_30D": surv[:, 0],
    "SURV_PROB_60D": surv[:, 1],
    "SURV_PROB_90D": surv[:, 2],
})


assert len(sub) == 255000, f"Expected 255000 rows, got {len(sub)}"
assert (sub.SURV_PROB_30D >= sub.SURV_PROB_60D).all(), "Monotonicity violated 30>=60"
assert (sub.SURV_PROB_60D >= sub.SURV_PROB_90D).all(), "Monotonicity violated 60>=90"
assert sub[["SURV_PROB_30D","SURV_PROB_60D","SURV_PROB_90D"]].apply(
    lambda s: s.between(0, 1)).all().all(), "Survival probs out of [0,1]"

sub.to_csv("prediction.csv", index=False)
print(f"Wrote prediction.csv  {sub.shape}")
print(sub.describe().round(4))
print(sub.head())

Wrote prediction.csv  (255000, 5)
        RISK_SCORE  SURV_PROB_30D  SURV_PROB_60D  SURV_PROB_90D
count  255000.0000    255000.0000    255000.0000    255000.0000
mean        0.0555         0.9996         0.9969         0.9480
std         0.0605         0.0019         0.0083         0.0538
min         0.0022         0.8755         0.7806         0.0661
25%         0.0155         1.0000         0.9992         0.9276
50%         0.0341         1.0000         0.9997         0.9666
75%         0.0750         1.0000         0.9998         0.9849
max         1.1474         1.0000         0.9999         0.9980
     ACCOUNT_ID  RISK_SCORE  SURV_PROB_30D  SURV_PROB_60D  SURV_PROB_90D
0  CUST00317794    0.014629       0.999983       0.999839       0.985548
1  CUST00635595    0.119920       0.999983       0.999633       0.880464
2  CUST00438228    0.008314       0.999984       0.999846       0.991856
3  CUST00189869    0.005678       0.999979       0.999742       0.994602
4  CUST00307108    0.0080